#  Laboratorio 5: ECG

Importar librerías

In [72]:
!pip install neurokit2 -q

In [73]:
import os, glob, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, iirnotch, find_peaks
import neurokit2 as nk

Configuración

In [74]:
ROOT = "/content"               # Carpeta con los .txt
FS = 1000                       # Frecuencia de muestreo (Hz)
NOTCH = 60                      # 60 Hz

def ensure_dir(p):
    os.makedirs(p, exist_ok=True)
    return p

OUT = ensure_dir(os.path.join(ROOT, "outputs"))
OUT_RAW_T = ensure_dir(os.path.join(OUT, "raw", "time"))      # (ya no se usa, pero se mantiene por si acaso)
OUT_RAW_F = ensure_dir(os.path.join(OUT, "raw", "freq"))
OUT_FILT_T = ensure_dir(os.path.join(OUT, "filtered", "time"))
OUT_FILT_F = ensure_dir(os.path.join(OUT, "filtered", "freq"))
OUT_PIPELINE = ensure_dir(os.path.join(OUT, "pipeline"))
OUT_COMPARE = ensure_dir(os.path.join(OUT, "comparison"))     # nueva carpeta para la figura compuesta


Carga de señal

In [75]:
def cargar_senal(ruta):
    """Lee archivo .txt con números; última columna = ECG."""
    try:
        dat = np.loadtxt(ruta)
    except:
        dat = pd.read_csv(ruta, header=None, sep=None, engine='python').values
    if dat.ndim == 1:
        dat = dat.reshape(-1, 1)
    senal = dat[:, -1]                     # última columna
    # Escalado si valores > 2000 (ADC sin convertir)
    if np.max(np.abs(senal)) > 2000:
        senal = (senal - np.mean(senal)) / 1000.0
    t = np.arange(len(senal)) / FS
    print(f"   Señal: {len(senal)} muestras, duración {len(senal)/FS:.2f}s, fs={FS} Hz")
    return t, senal


Filtros

In [76]:
def notch(x):
    if FS <= 0 or NOTCH >= FS/2:
        return x
    b, a = iirnotch(NOTCH/(FS/2), Q=30)
    return filtfilt(b, a, x)

def bp_ecg(x):
    nyq = 0.5 * FS
    low = max(1e-6, 0.5/nyq)
    high = min(0.999, 40.0/nyq)
    if high <= low:
        return x
    b, a = butter(4, [low, high], btype='band')
    return filtfilt(b, a, x)

def espectro(x):
    if FS <= 0 or len(x) < 2:
        return None, None
    x = x - np.mean(x)
    w = np.hanning(len(x))
    X = np.fft.rfft(x * w)
    f = np.fft.rfftfreq(len(x), 1/FS)
    mag = np.abs(X) / np.sum(w)
    return f, mag

Detección de picos R

In [77]:
def detectar_picos(senal_filtrada):
    s_norm = (senal_filtrada - np.mean(senal_filtrada)) / (np.std(senal_filtrada) + 1e-8)
    try:
        _, info = nk.ecg_peaks(s_norm, sampling_rate=FS, method='neurokit')
        picos = np.where(info['ECG_R_Peaks'])[0]
        if len(picos) > 1:
            hr = 60.0 / (np.mean(np.diff(picos)) / FS)
            if 30 <= hr <= 200:
                return picos
    except:
        pass
    dist_min = int(0.22 * FS)
    prominencia = 0.3 * np.std(s_norm)
    picos, _ = find_peaks(s_norm, distance=dist_min, prominence=prominencia)
    if len(picos) == 0:
        altura = np.percentile(s_norm, 95)
        picos, _ = find_peaks(s_norm, distance=dist_min, height=altura)
    if len(picos) > 1:
        keep = [picos[0]]
        for i in range(1, len(picos)):
            if picos[i] - keep[-1] >= int(0.18 * FS):
                keep.append(picos[i])
        picos = np.array(keep, dtype=int)
    return picos


Métricas

In [78]:
def metricas(picos):
    if len(picos) < 2:
        return {"n_R":0, "HR_media":np.nan, "HR_min":np.nan, "HR_max":np.nan,
                "SDNN":np.nan, "RMSSD":np.nan, "pNN50":np.nan}
    rr = np.diff(picos) / FS
    rr_valid = rr[(rr >= 0.3) & (rr <= 1.5)]
    if len(rr_valid) < 2:
        return {"n_R":0, "HR_media":np.nan, "HR_min":np.nan, "HR_max":np.nan,
                "SDNN":np.nan, "RMSSD":np.nan, "pNN50":np.nan}
    hr_inst = 60.0 / rr_valid
    drr = np.diff(rr_valid)
    return {
        "n_R": len(picos),
        "HR_media": np.mean(hr_inst),
        "HR_min": np.min(hr_inst),
        "HR_max": np.max(hr_inst),
        "SDNN": np.std(rr_valid, ddof=1),
        "RMSSD": np.sqrt(np.mean(drr**2)) if len(drr) > 0 else np.nan,
        "pNN50": np.mean(np.abs(drr) > 0.05) * 100.0 if len(drr) > 0 else np.nan
    }


Gráficas

In [79]:
def guardar_tiempo(t, y, titulo, ruta):
    plt.figure(figsize=(12,4))
    plt.plot(t, y, linewidth=0.8)
    plt.xlabel("Tiempo (s)"); plt.ylabel("Amplitud (mV)")
    plt.title(titulo); plt.tight_layout()
    plt.savefig(ruta, dpi=150); plt.show(); plt.close()

def guardar_con_picos(t, y, picos, titulo, ruta):
    plt.figure(figsize=(12,4))
    plt.plot(t, y, linewidth=0.8)
    if len(picos) > 0:
        plt.plot(t[picos], y[picos], 'ro', markersize=3, label='Picos R')
        plt.legend()
    plt.xlabel("Tiempo (s)"); plt.ylabel("Amplitud (mV)")
    plt.title(titulo); plt.tight_layout()
    plt.savefig(ruta, dpi=150); plt.show(); plt.close()

def guardar_espectro(f, mag, titulo, ruta):
    if f is None:
        return
    plt.figure(figsize=(12,4))
    plt.plot(f, mag, linewidth=0.8)
    plt.xlim(0, min(80, f.max()))
    plt.xlabel("Frecuencia (Hz)"); plt.ylabel("Magnitud (rel.)")
    plt.title(titulo); plt.tight_layout()
    plt.savefig(ruta, dpi=150); plt.show(); plt.close()

def guardar_tabla(met, duracion, n_muestras, titulo, ruta):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.axis('off')
    datos = [
        ["Parámetro", "Valor"],
        ["Nº picos R", f"{met['n_R']}"],
        ["HR media (lpm)", f"{met['HR_media']:.1f}" if not np.isnan(met['HR_media']) else "NA"],
        ["HR mínima (lpm)", f"{met['HR_min']:.1f}" if not np.isnan(met['HR_min']) else "NA"],
        ["HR máxima (lpm)", f"{met['HR_max']:.1f}" if not np.isnan(met['HR_max']) else "NA"],
        ["SDNN (s)", f"{met['SDNN']:.3f}" if not np.isnan(met['SDNN']) else "NA"],
        ["RMSSD (s)", f"{met['RMSSD']:.3f}" if not np.isnan(met['RMSSD']) else "NA"],
        ["pNN50 (%)", f"{met['pNN50']:.1f}" if not np.isnan(met['pNN50']) else "NA"],
        ["Duración (s)", f"{duracion:.2f}"],
        ["Muestras", f"{n_muestras}"],
        ["Fs (Hz)", f"{FS}"]
    ]
    tabla = ax.table(cellText=datos, loc='center', cellLoc='center', colWidths=[0.3, 0.5])
    tabla.auto_set_font_size(False)
    tabla.set_fontsize(10)
    tabla.scale(1.2, 1.5)
    for (i, j), celda in tabla.get_celld().items():
        if i == 0:
            celda.set_facecolor('#40466e')
            celda.set_text_props(weight='bold', color='white')
        else:
            celda.set_facecolor('#f5f5f5')
    plt.title(titulo, fontsize=12, weight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(ruta, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

def guardar_comparativa(t, senal_cruda, senal_filtrada, titulo, ruta):
    """Crea una figura con 4 subplots: cruda tiempo, cruda espectro, filtrada tiempo, filtrada espectro."""
    f_cruda, mag_cruda = espectro(senal_cruda)
    f_filt, mag_filt = espectro(senal_filtrada)

    fig, axs = plt.subplots(2, 2, figsize=(14, 10))
    # Señal cruda
    axs[0,0].plot(t, senal_cruda, linewidth=0.8, color='b')
    axs[0,0].set_xlabel("Tiempo (s)")
    axs[0,0].set_ylabel("Amplitud (mV)")
    axs[0,0].set_title("Señal ECG cruda")
    axs[0,0].grid(True, linestyle='--', alpha=0.5)

    # Espectro crudo
    if f_cruda is not None:
        axs[0,1].plot(f_cruda, mag_cruda, linewidth=0.8, color='b')
        axs[0,1].set_xlim(0, min(80, f_cruda.max()))
        axs[0,1].set_xlabel("Frecuencia (Hz)")
        axs[0,1].set_ylabel("Magnitud (rel.)")
        axs[0,1].set_title("Espectro (crudo)")
        axs[0,1].grid(True, linestyle='--', alpha=0.5)

    # Señal filtrada
    axs[1,0].plot(t, senal_filtrada, linewidth=0.8, color='r')
    axs[1,0].set_xlabel("Tiempo (s)")
    axs[1,0].set_ylabel("Amplitud (mV)")
    axs[1,0].set_title("Señal ECG filtrada (0.5-40 Hz + notch 60 Hz)")
    axs[1,0].grid(True, linestyle='--', alpha=0.5)

    # Espectro filtrado
    if f_filt is not None:
        axs[1,1].plot(f_filt, mag_filt, linewidth=0.8, color='r')
        axs[1,1].set_xlim(0, min(80, f_filt.max()))
        axs[1,1].set_xlabel("Frecuencia (Hz)")
        axs[1,1].set_ylabel("Magnitud (rel.)")
        axs[1,1].set_title("Espectro (filtrado)")
        axs[1,1].grid(True, linestyle='--', alpha=0.5)

    plt.suptitle(titulo, fontsize=14, weight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(ruta, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()


Pipeline pricipal

In [80]:
archivos = sorted(glob.glob(os.path.join(ROOT, "*.txt")))
print(f"Archivos encontrados: {len(archivos)}")
for a in archivos:
    print("   -", os.path.basename(a))

for ruta in archivos:
    base = os.path.splitext(os.path.basename(ruta))[0]
    print(f"\nProcesando: {base}")
    try:
        t, senal = cargar_senal(ruta)
        if np.std(senal) < 1e-6:
            print("Señal constante. Omitiendo.\n")
            continue

        # FILTRADO
        senal_f = notch(senal)
        senal_f = bp_ecg(senal_f)

        # FIGURA COMPARATIVA
        guardar_comparativa(t, senal, senal_f,
                            f"Comparativa - {base}",
                            os.path.join(OUT_COMPARE, f"{base}_comparison.png"))

        # RECORTE 20-25 s
        if len(senal_f) > 25*FS:
            i0, i1 = int(20*FS), int(25*FS)
            guardar_tiempo(t[i0:i1], senal_f[i0:i1], f"{base} - ECG (filtrado, 20-25 s)",
                           os.path.join(OUT_FILT_T, f"{base}_time_filt_crop.png"))

        # DETECCIÓN DE PICOS R Y MÉTRICAS
        picos = detectar_picos(senal_f)
        print(f"Picos R detectados: {len(picos)}")
        met = metricas(picos)

        #Señal filtrada con picos R
        guardar_con_picos(t, senal_f, picos, f"{base} - ECG (picos R)",
                          os.path.join(OUT_PIPELINE, f"{base}_peaks.png"))

        # FC instantánea
        if len(picos) > 1:
            rr = np.diff(picos) / FS
            hr_inst = 60.0 / rr
            mask = (hr_inst >= 30) & (hr_inst <= 200)
            if np.any(mask):
                t_rr = (t[picos[1:]] + t[picos[:-1]]) / 2.0
                guardar_tiempo(t_rr[mask], hr_inst[mask], f"{base} - ECG (FC instantánea)",
                               os.path.join(OUT_PIPELINE, f"{base}_hr_inst.png"))

        # TABLA RESUMEN (solo señal completa)
        duracion = len(senal) / FS
        guardar_tabla(met, duracion, len(senal), f"Resumen - {base}",
                      os.path.join(OUT_PIPELINE, f"{base}_summary.png"))

    except Exception as e:
        print(f"Error: {e}")

print("\nProcesamiento completado. Resultados en /content/outputs/")

Output hidden; open in https://colab.research.google.com to view.